In [2]:
"""
# AI Phishing Detection System - Phase 1 Model Training

## Objective

This notebook trains a machine learning model to classify URLs as phishing or legitimate.

## Dataset

Dataset:
- PhiUSIIL Phishing URL Dataset
- UCI Machine Learning Repository (ID: 967)
- 235,795 URLs
- 54 URL-based features

## Label Convention

- 0 = Phishing
- 1 = Legitimate

## Model

Decision Tree Classifier with:
- GridSearchCV hyperparameter tuning
- Model evaluation
- Feature importance analysis
- Robustness testing
"""

'\n# AI Phishing Detection System - Phase 1 Model Training\n\n## Objective\n\nThis notebook trains a machine learning model to classify URLs as phishing or legitimate.\n\n## Dataset\n\nDataset:\n- PhiUSIIL Phishing URL Dataset\n- UCI Machine Learning Repository (ID: 967)\n- 235,795 URLs\n- 54 URL-based features\n\n## Label Convention\n\n- 0 = Phishing\n- 1 = Legitimate\n\n## Model\n\nDecision Tree Classifier with:\n- GridSearchCV hyperparameter tuning\n- Model evaluation\n- Feature importance analysis\n- Robustness testing\n'

In [3]:

import os
import warnings

import joblib
import matplotlib

matplotlib.use("Agg")  # headless environments (servers, CI)
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

In [4]:
CSV_LOCAL = "PhiUSIIL_Phishing_URL_Dataset.csv"
OUTPUT_DIR = "outputs"
RANDOM_STATE = 42
TEST_SIZE = 0.20


DROP_COLS = ["FILENAME", "URL", "Domain", "TLD", "Title"]

os.makedirs(OUTPUT_DIR, exist_ok=True)

In [5]:
def savefig(name: str) -> None:
    """Save the current matplotlib figure into the outputs directory."""
    path = os.path.join(OUTPUT_DIR, name)
    plt.tight_layout()
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"   [saved] {path}")

In [ ]:
print("=" * 70)
print("[1/7] Loading dataset ...")
print("=" * 70)

if os.path.exists(CSV_LOCAL):
    df = pd.read_csv(CSV_LOCAL, encoding="utf-8")
    print(f"   Loaded local CSV: {CSV_LOCAL}")
else:
    # Fallback: fetch from the UCI Machine Learning Repository
    from ucimlrepo import fetch_ucirepo

    print("   Local CSV not found - fetching from UCI (id=967) ...")
    phiusiil = fetch_ucirepo(id=967)
    df = pd.concat([phiusiil.data.features, phiusiil.data.targets], axis=1)

print(f"   Shape: {df.shape[0]:,} rows x {df.shape[1]} columns")

In [ ]:
print("\n" + "=" * 70)
print("[2/7] Preprocessing ...")
print("=" * 70)

# 2a. Drop identifier / free-text columns
present_drops = [c for c in DROP_COLS if c in df.columns]
df = df.drop(columns=present_drops)
print(f"   Dropped columns: {present_drops}")

# 2b. Duplicates
n_dupes = df.duplicated().sum()
if n_dupes:
    df = df.drop_duplicates().reset_index(drop=True)
print(f"   Duplicate rows removed: {n_dupes}")

# 2c. Missing values
n_missing = df.isnull().sum().sum()
if n_missing:
    df = df.dropna().reset_index(drop=True)
print(f"   Missing values handled: {n_missing}")

X = df.drop(columns=["label"])
y = df["label"]
print(f"   Feature matrix: {X.shape[1]} numeric features")

# 2d. Stratified 80/20 split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)
print(f"   Train: {X_train.shape[0]:,}  |  Test: {X_test.shape[0]:,}  (stratified)")